#  Feature Selection dengan Information Gain
### Implementasi Python — Tanpa Library Eksternal

---

> **Dataset:** Prediksi Keputusan Membeli Laptop — 14 baris, 4 fitur, 2 kelas

---
## BAGIAN 1 — Load Dataset dari CSV

In [ ]:
import csv
import math

def load_csv(filepath):
    """
    Membaca file CSV secara manual menggunakan modul csv bawaan Python.
    Mengembalikan: (header, data)
    """
    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        rows = list(reader)
    header = rows[0]    =
    data   = rows[1:]    
    return header, data

# Load
FILEPATH = 'dataset_laptop.csv'
header, dataset = load_csv(FILEPATH)

FITUR     = header[:-1]   
LABEL_COL = header[-1]    
LABEL_IDX = len(FITUR)    

print('=' * 55)
print('  Dataset berhasil dimuat!')
print('=' * 55)
print(f'  File      : {FILEPATH}')
print(f'  Fitur     : {FITUR}')
print(f'  Label     : {LABEL_COL}')
print(f'  Jumlah    : {len(dataset)} baris')

  Dataset berhasil dimuat!
  File      : dataset_laptop.csv
  Fitur     : ['Penghasilan', 'Usia_Kategori', 'Status_Pekerjaan', 'Punya_Tabungan']
  Label     : Beli_Laptop
  Jumlah    : 14 baris


In [ ]:
def tampilkan_tabel(header, data):
    lebar = [max(len(str(row[i])) for row in [header] + data) + 2 for i in range(len(header))]
    garis = '+' + '+'.join('-' * w for w in lebar) + '+'
    def format_row(row):
        return '|' + '|'.join(f' {str(v):<{w-1}}' for v, w in zip(row, lebar)) + '|'

    print(garis)
    print(format_row(['No'] + header))
    print(garis)
    for i, row in enumerate(data, 1):
        print(format_row([i] + row))
    print(garis)
    print(f'  Total: {len(data)} baris')

tampilkan_tabel(header, dataset)

+-------------+---------------+------------------+----------------+-------------+
| No          | Penghasilan   | Usia_Kategori    | Status_Pekerjaan| Punya_Tabungan|
+-------------+---------------+------------------+----------------+-------------+
| 1           | Tinggi        | Muda             | Karyawan       | Ya          |
| 2           | Tinggi        | Muda             | Karyawan       | Tidak       |
| 3           | Menengah      | Tua              | Karyawan       | Ya          |
| 4           | Rendah        | Tua              | Wirausaha      | Ya          |
| 5           | Rendah        | Tua              | Wirausaha      | Tidak       |
| 6           | Rendah        | Menengah         | Wirausaha      | Tidak       |
| 7           | Menengah      | Menengah         | Wirausaha      | Tidak       |
| 8           | Tinggi        | Muda             | Karyawan       | Tidak       |
| 9           | Tinggi        | Tua              | Wirausaha      | Ya          |
| 10         

In [ ]:
distribusi = {}
for baris in dataset:
    label = baris[LABEL_IDX]
    distribusi[label] = distribusi.get(label, 0) + 1

total = len(dataset)
print('Distribusi Kelas Label:')
print('-' * 35)
for label, jumlah in distribusi.items():
    pct = jumlah / total * 100
    bar = '█' * int(pct / 5)
    print(f'  {label:<8}: {jumlah:>2} data ({pct:.1f}%)  {bar}')
print('-' * 35)
print(f'  Total   : {total} data')

Distribusi Kelas Label:
-----------------------------------
  Ya      :  9 data (64.3%)  ████████████
  Tidak   :  5 data (35.7%)  ███████
-----------------------------------
  Total   : 14 data


---
##  BAGIAN 2 — Implementasi Fungsi Entropy


In [ ]:
def hitung_entropy(data):
    """
    Menghitung entropy H(S) dari himpunan data.

    Rumus:
        H(S) = - SUM [ p(c) * log2(p(c)) ]

    Parameter:
        data (list): Daftar baris data.

    Return:
        float: Nilai entropy (0 = murni, mendekati 1 = tidak murni)
    """
    if len(data) == 0:
        return 0.0
  
    frekuensi = {}
    for baris in data:
        label = baris[LABEL_IDX]
        frekuensi[label] = frekuensi.get(label, 0) + 1

    total   = len(data)
    entropy = 0.0
    for label, jumlah in frekuensi.items():
        p        = jumlah / total          # probabilitas kelas
        entropy -= p * math.log2(p)        # akumulasi

    return entropy


H_S = hitung_entropy(dataset)

print('=' * 55)
print('  ENTROPY KESELURUHAN  H(S)')
print('=' * 55)
print(f'  Distribusi : {distribusi}')
print(f'  Total data : {total}')
print()

for label, jumlah in distribusi.items():
    p = jumlah / total
    kontribusi = -p * math.log2(p)
    print(f'  p({label}) = {jumlah}/{total} = {p:.4f}')
    print(f'  kontribusi = -({p:.4f}) * log2({p:.4f}) = {kontribusi:.6f}')
    print()

print(f'  H(S) = {H_S:.6f}')
print('=' * 55)
print(f'  Interpretasi: Nilai {H_S:.4f} ≈ 0.94 mendekati 1')
print(f'  → Dataset cukup tidak murni (kedua kelas cukup berimbang)')

  ENTROPY KESELURUHAN  H(S)
  Distribusi : {'Ya': 9, 'Tidak': 5}
  Total data : 14

  p(Ya) = 9/14 = 0.6429
  kontribusi = -(0.6429) * log2(0.6429) = 0.409776

  p(Tidak) = 5/14 = 0.3571
  kontribusi = -(0.3571) * log2(0.3571) = 0.530510

  H(S) = 0.940286
  Interpretasi: Nilai 0.9403 ≈ 0.94 mendekati 1
  → Dataset cukup tidak murni (kedua kelas cukup berimbang)


---
##  BAGIAN 3 — Implementasi Fungsi Information Gain


In [ ]:
def hitung_information_gain(data, indeks_fitur):
    """
    Menghitung Information Gain IG(S, A) untuk satu fitur.

    Rumus:
        IG(S, A) = H(S) - SUM_v [ (|S_v| / |S|) * H(S_v) ]

    Parameter:
        data          (list): Himpunan data lengkap.
        indeks_fitur  (int) : Indeks kolom fitur yang dievaluasi.

    Return:
        float: Nilai Information Gain.
    """
    entropy_awal = hitung_entropy(data)
    total        = len(data)

    subset_per_nilai = {}
    for baris in data:
        nilai = baris[indeks_fitur]
        if nilai not in subset_per_nilai:
            subset_per_nilai[nilai] = []
        subset_per_nilai[nilai].append(baris)

    entropy_terbobot = 0.0
    for nilai, subset in subset_per_nilai.items():
        bobot             = len(subset) / total
        entropy_terbobot += bobot * hitung_entropy(subset)

    information_gain = entropy_awal - entropy_terbobot
    return information_gain

print('Uji Cepat Information Gain:')
print('-' * 45)
for i, nama in enumerate(FITUR):
    ig = hitung_information_gain(dataset, i)
    print(f'  IG(S, {nama:<20}) = {ig:.6f}')

Uji Cepat Information Gain:
---------------------------------------------
  IG(S, Penghasilan         ) = 0.103893
  IG(S, Usia_Kategori       ) = 0.403873
  IG(S, Status_Pekerjaan    ) = 0.016112
  IG(S, Punya_Tabungan      ) = 0.236122


---
##  BAGIAN 4 — Detail Perhitungan Per Fitur

In [ ]:
def tampilkan_detail_ig(data, indeks_fitur, nama_fitur):
    """
    Menampilkan setiap langkah perhitungan IG untuk satu fitur
    secara transparan dan terdokumentasi.
    """
    entropy_awal = hitung_entropy(data)
    total        = len(data)

    print(f'┌────────────────────────────────────────────────────┐')
    print(f'│  Fitur: {nama_fitur:<43}│')
    print(f'└────────────────────────────────────────────────────┘')
    print(f'  H(S) = {entropy_awal:.6f}   |S| = {total}')
    print()

    subset_per_nilai = {}
    for baris in data:
        nilai = baris[indeks_fitur]
        if nilai not in subset_per_nilai:
            subset_per_nilai[nilai] = []
        subset_per_nilai[nilai].append(baris)

    print(f'  {"Nilai":<12} {"Distribusi Kelas":<22} {"Bobot":>7} {"H(Sv)":>10} {"Bobot×H(Sv)":>12}')
    print(f'  {"─"*72}')

    entropy_terbobot_total = 0.0
    for nilai, subset in subset_per_nilai.items():
        n_sub  = len(subset)
        bobot  = n_sub / total
        ent    = hitung_entropy(subset)
        kontri = bobot * ent
        entropy_terbobot_total += kontri

        dist = {}
        for b in subset:
            lbl = b[LABEL_IDX]
            dist[lbl] = dist.get(lbl, 0) + 1
        dist_str = ', '.join(f'{k}={v}' for k, v in dist.items())

        print(f'  {nilai:<12} [{dist_str}] {n_sub}/{total}{"":>3} {bobot:>7.4f} {ent:>10.6f} {kontri:>12.6f}')

    print(f'  {"─"*72}')
    ig = entropy_awal - entropy_terbobot_total
    print(f'  Entropy terbobot Σ(|Sv|/|S|)·H(Sv)   = {entropy_terbobot_total:.6f}')
    print(f'  IG = H(S) - Σ = {entropy_awal:.6f} - {entropy_terbobot_total:.6f}')
    print(f'  IG({nama_fitur}) = {ig:.6f}')
    print()
    return ig

In [ ]:
ig_usia = tampilkan_detail_ig(dataset, 1, 'Usia_Kategori')

┌────────────────────────────────────────────────────┐
│  Fitur: Usia_Kategori                              │
└────────────────────────────────────────────────────┘
  H(S) = 0.940286   |S| = 14

  Nilai        Distribusi Kelas         Bobot      H(Sv)  Bobot×H(Sv)
  ────────────────────────────────────────────────────────────────────────
  Muda         [Ya=5, Tidak=1] 6/14     0.4286   0.650022     0.278581
  Tua          [Ya=4, Tidak=1] 5/14     0.3571   0.721928     0.257831
  Menengah     [Tidak=3] 3/14     0.2143   0.000000     0.000000
  ────────────────────────────────────────────────────────────────────────
  Entropy terbobot Σ(|Sv|/|S|)·H(Sv)   = 0.536413
  IG = H(S) - Σ = 0.940286 - 0.536413
  IG(Usia_Kategori) = 0.403873



In [ ]:
ig_tabungan = tampilkan_detail_ig(dataset, 3, 'Punya_Tabungan')

┌────────────────────────────────────────────────────┐
│  Fitur: Punya_Tabungan                             │
└────────────────────────────────────────────────────┘
  H(S) = 0.940286   |S| = 14

  Nilai        Distribusi Kelas         Bobot      H(Sv)  Bobot×H(Sv)
  ────────────────────────────────────────────────────────────────────────
  Ya           [Ya=7, Tidak=1] 8/14     0.5714   0.543564     0.310608
  Tidak        [Ya=2, Tidak=4] 6/14     0.4286   0.918296     0.393555
  ────────────────────────────────────────────────────────────────────────
  Entropy terbobot Σ(|Sv|/|S|)·H(Sv)   = 0.704164
  IG = H(S) - Σ = 0.940286 - 0.704164
  IG(Punya_Tabungan) = 0.236122



In [ ]:
ig_penghasilan = tampilkan_detail_ig(dataset, 0, 'Penghasilan')

┌────────────────────────────────────────────────────┐
│  Fitur: Penghasilan                                │
└────────────────────────────────────────────────────┘
  H(S) = 0.940286   |S| = 14

  Nilai        Distribusi Kelas         Bobot      H(Sv)  Bobot×H(Sv)
  ────────────────────────────────────────────────────────────────────────
  Tinggi       [Ya=4, Tidak=1] 5/14     0.3571   0.721928     0.257831
  Menengah     [Ya=3, Tidak=1] 4/14     0.2857   0.811278     0.231794
  Rendah       [Ya=2, Tidak=3] 5/14     0.3571   0.970951     0.346768
  ────────────────────────────────────────────────────────────────────────
  Entropy terbobot Σ(|Sv|/|S|)·H(Sv)   = 0.836393
  IG = H(S) - Σ = 0.940286 - 0.836393
  IG(Penghasilan) = 0.103893



In [ ]:
ig_status = tampilkan_detail_ig(dataset, 2, 'Status_Pekerjaan')

┌────────────────────────────────────────────────────┐
│  Fitur: Status_Pekerjaan                           │
└────────────────────────────────────────────────────┘
  H(S) = 0.940286   |S| = 14

  Nilai        Distribusi Kelas         Bobot      H(Sv)  Bobot×H(Sv)
  ────────────────────────────────────────────────────────────────────────
  Karyawan     [Ya=5, Tidak=2] 7/14     0.5000   0.863121     0.431560
  Wirausaha    [Ya=4, Tidak=3] 7/14     0.5000   0.985228     0.492614
  ────────────────────────────────────────────────────────────────────────
  Entropy terbobot Σ(|Sv|/|S|)·H(Sv)   = 0.924174
  IG = H(S) - Σ = 0.940286 - 0.924174
  IG(Status_Pekerjaan) = 0.016112



---
##  BAGIAN 5 — Feature Selection & Ranking

In [ ]:
def feature_selection(data, fitur):
    """
    Menghitung IG semua fitur dan mengurutkan dari IG tertinggi ke terendah.

    Parameter:
        data  (list): Dataset lengkap.
        fitur (list): Daftar nama fitur.

    Return:
        list: [(nama_fitur, ig_value), ...] terurut descending.
    """
    hasil = []
    for i, nama_fitur in enumerate(fitur):
        ig = hitung_information_gain(data, i)
        hasil.append((nama_fitur, ig))

    hasil.sort(key=lambda x: x[1], reverse=True)
    return hasil

ranking = feature_selection(dataset, FITUR)
max_ig  = ranking[0][1]

print('=' * 60)
print('  RANKING FITUR — INFORMATION GAIN')
print('=' * 60)
print(f'  {"Rank":<6} {"Nama Fitur":<22} {"IG":>10}  Visualisasi')
print(f'  {"─"*58}')

label_rank = ['★ 1st', '  2nd', '  3rd', '  4th']
for i, (nama, ig) in enumerate(ranking):
    bar = '█' * int(ig / max_ig * 30)
    print(f'  {label_rank[i]}  {nama:<22} {ig:>10.6f}  {bar}')

print('=' * 60)
print(f'  Fitur terpilih (IG tertinggi): "{ranking[0][0]}"')
print(f'  → Digunakan sebagai ROOT NODE pada Decision Tree')

  RANKING FITUR — INFORMATION GAIN
  Rank   Nama Fitur                     IG  Visualisasi
  ──────────────────────────────────────────────────────────
  ★ 1st  Usia_Kategori            0.403873  ██████████████████████████████
    2nd  Punya_Tabungan           0.236122  █████████████████
    3rd  Penghasilan              0.103893  ███████
    4th  Status_Pekerjaan         0.016112  █
  Fitur terpilih (IG tertinggi): "Usia_Kategori"
  → Digunakan sebagai ROOT NODE pada Decision Tree


---
##  BAGIAN 6 — Visualisasi

In [ ]:
def ascii_bar_chart(data_dict, judul, lebar=50):
    print(f'\n  {judul}')
    print(f'  {"─" * (lebar + 28)}')
    max_val = max(data_dict.values())
    for nama, val in data_dict.items():
        panjang_bar = int(val / max_val * lebar)
        bar = '▓' * panjang_bar + '░' * (lebar - panjang_bar)
        print(f'  {nama:<22} |{bar}| {val:.6f}')
    print(f'  {"─" * (lebar + 28)}')

ig_dict = {nama: ig for nama, ig in ranking}
ascii_bar_chart(ig_dict, 'Information Gain per Fitur (▓ = proporsi IG)')

print()
print('  Nilai Entropy Keseluruhan H(S) = 0.940286')
print()
print('  Semakin tinggi IG → semakin besar kemampuan fitur')
print('  memisahkan kelas dalam dataset.')


  Information Gain per Fitur (▓ = proporsi IG)
  ──────────────────────────────────────────────────────────────────────────────
  Usia_Kategori          |▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓| 0.403873
  Punya_Tabungan         |▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░| 0.236122
  Penghasilan            |▓▓▓▓▓▓▓▓▓▓▓▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 0.103893
  Status_Pekerjaan       |▓░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░| 0.016112
  ──────────────────────────────────────────────────────────────────────────────

  Nilai Entropy Keseluruhan H(S) = 0.940286

  Semakin tinggi IG → semakin besar kemampuan fitur
  memisahkan kelas dalam dataset.


---
##  BAGIAN 7 — Pipeline Lengkap (End-to-End)

In [ ]:
def jalankan_feature_selection(filepath):
    """
    Pipeline end-to-end Feature Selection menggunakan Information Gain:
    1. Load CSV
    2. Hitung Entropy keseluruhan
    3. Hitung IG setiap fitur
    4. Ranking fitur
    5. Cetak hasil
    """
    header, data = load_csv(filepath)
    global LABEL_IDX
    fitur_list = header[:-1]
    LABEL_IDX  = len(fitur_list)

    print('╔══════════════════════════════════════════════════════╗')
    print('║      FEATURE SELECTION — INFORMATION GAIN           ║')
    print('╚══════════════════════════════════════════════════════╝')
    print(f'  Dataset  : {filepath}')
    print(f'  Fitur    : {fitur_list}')
    print(f'  Label    : {header[-1]}')
    print(f'  Jumlah   : {len(data)} baris')

    H_S = hitung_entropy(data)
    print(f'\n  H(S) Keseluruhan  = {H_S:.6f}')

    hasil = feature_selection(data, fitur_list)

    print()
    print(f'  {"Rank":<5} {"Fitur":<22} {"IG":>10}  {"Status"}')
    print(f'  {"─"*55}')
    for rank, (nama, ig) in enumerate(hasil, 1):
        status = '← FITUR TERBAIK (Root Node)' if rank == 1 else ''
        print(f'  {rank:<5} {nama:<22} {ig:>10.6f}  {status}')
    print(f'  {"─"*55}')
    print(f'  H(S) awal = {H_S:.6f}')

    return hasil

hasil_akhir = jalankan_feature_selection('dataset_laptop.csv')

╔══════════════════════════════════════════════════════╗
║      FEATURE SELECTION — INFORMATION GAIN           ║
╚══════════════════════════════════════════════════════╝
  Dataset  : dataset_laptop.csv
  Fitur    : ['Penghasilan', 'Usia_Kategori', 'Status_Pekerjaan', 'Punya_Tabungan']
  Label    : Beli_Laptop
  Jumlah   : 14 baris

  H(S) Keseluruhan  = 0.940286

  Rank  Fitur                          IG  Status
  ───────────────────────────────────────────────────────
  1     Usia_Kategori            0.403873  ← FITUR TERBAIK (Root Node)
  2     Punya_Tabungan           0.236122  
  3     Penghasilan              0.103893  
  4     Status_Pekerjaan         0.016112  
  ───────────────────────────────────────────────────────
  H(S) awal = 0.940286


---
## BAGIAN 8 — Kesimpulan

Berdasarkan hasil perhitungan Information Gain:

| Rank | Fitur | IG | Interpretasi |
|------|-------|----|--------------|
| 1 ★ | **Usia_Kategori** | 0.403873 | Tertinggi → Root Node Decision Tree |
| 2 | Punya_Tabungan | 0.236122 | Cukup informatif |
| 3 | Penghasilan | 0.103893 | Informatif moderat |
| 4 | Status_Pekerjaan | 0.016112 | Hampir tidak informatif |

### Mengapa Usia_Kategori terbaik?
Ketika data dibagi berdasarkan usia:
- Kelompok **Menengah** → entropy = **0.0** (semua 3 data = 'Tidak')  sangat murni
- Kelompok **Muda** → 5 Ya dari 6 data (cukup murni)
- Kelompok **Tua** → 4 Ya dari 5 data (cukup murni)

Pembagian ini menghasilkan subset yang jauh lebih murni dari entropy awal (0.9403).
